# Bond Valuation Engine Validation

Yield Curve 기반 채권 공정가치 평가 및 가격 검증 프로젝트

이 노트북은 공개 시장금리와 단순화된 평가모형을 이용해 채권의 기준가격을 산출하고, 금리 변화에 따른 가격민감도와 가격 검증 로직을 구현하는 학부 수준의 포트폴리오 프로젝트입니다.

주의:
- `yield_curve.csv`의 금리는 프로젝트 구조를 실행하기 위한 예시 데이터입니다.
- 실제 제출용 최종본에서는 한국은행 ECOS, 금융투자협회, FRED 등 공식 공개 데이터로 교체하는 것이 좋습니다.
- 아래 Engine Price는 실제 평가사 가격이 아니라 검증 절차를 보여주기 위한 가상 값입니다.


## 1. 라이브러리 불러오기


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Yield Curve 데이터 읽기

CSV 파일의 `yield_pct`는 퍼센트 단위이므로 계산에서는 100으로 나누어 소수형 금리로 변환합니다.


In [ ]:
DATA_PATH = Path("yield_curve.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../yield_curve.csv")

curve = pd.read_csv(DATA_PATH)
curve["yield"] = curve["yield_pct"] / 100
curve


## 3. Yield Curve 시각화


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(curve["tenor_years"], curve["yield_pct"], marker="o")
plt.xlabel("Maturity (Years)")
plt.ylabel("Yield (%)")
plt.title("Sample Yield Curve")
plt.grid(alpha=0.3)
plt.show()


## 4. 평가대상 채권 설정

3년, 5년, 10년 채권을 비교하여 만기에 따른 금리민감도 차이를 확인합니다. Credit Spread는 평가구조를 설명하기 위한 단순화된 가정입니다.


In [ ]:
bonds = pd.DataFrame({
    "bond_id": ["BOND_A", "BOND_B", "BOND_C"],
    "face_value": [10000, 10000, 10000],
    "coupon_rate": [0.040, 0.035, 0.045],
    "maturity": [3, 5, 10],
    "frequency": [2, 2, 2],
    "credit_spread_bp": [40, 80, 120]
})

bonds


## 5. 채권가격 함수

각 쿠폰 현금흐름의 시점에 해당하는 금리를 선형보간하고, Credit Spread를 더한 뒤 현재가치로 할인합니다.

이 방식은 실제 평가사의 내부모형을 복제한 것이 아니라 Curve based pricing의 핵심 구조를 단순화하여 재현한 것입니다.


In [ ]:
def bond_price(face_value, coupon_rate, maturity, frequency, curve, credit_spread_bp=0):
    periods = int(maturity * frequency)
    times = np.arange(1, periods + 1) / frequency

    coupon = face_value * coupon_rate / frequency
    cashflows = np.full(periods, coupon, dtype=float)
    cashflows[-1] += face_value

    interpolated_rates = np.interp(times, curve["tenor_years"], curve["yield"])
    spread = credit_spread_bp / 10000
    discount_rates = interpolated_rates + spread
    discount_factors = 1 / (1 + discount_rates / frequency) ** (frequency * times)
    return float(np.sum(cashflows * discount_factors))


## 6. 독립 기준가격 산출


In [ ]:
bonds["reference_price"] = bonds.apply(
    lambda row: bond_price(
        row["face_value"], row["coupon_rate"], row["maturity"],
        row["frequency"], curve, row["credit_spread_bp"]
    ), axis=1
)

bonds[["bond_id", "maturity", "coupon_rate", "credit_spread_bp", "reference_price"]]


## 7. 가상 Engine Price와 비교

실제 평가사의 내부가격은 공개되어 있지 않으므로, 검증 로직 시연을 위해 기준가격에 작은 오차를 의도적으로 부여한 가상 Engine Price를 사용합니다.

0.30% 허용오차 역시 실제 회사의 내부 기준이 아니라 프로젝트용 가정입니다.


In [ ]:
bonds["engine_price"] = [
    bonds.loc[0, "reference_price"] * 1.0002,
    bonds.loc[1, "reference_price"] * 0.9960,
    bonds.loc[2, "reference_price"] * 1.0070
]

bonds["difference"] = bonds["engine_price"] - bonds["reference_price"]
bonds["difference_pct"] = bonds["difference"] / bonds["reference_price"] * 100

tolerance_pct = 0.30
bonds["validation"] = np.where(
    bonds["difference_pct"].abs() > tolerance_pct, "REVIEW", "PASS"
)

bonds[["bond_id", "reference_price", "engine_price", "difference_pct", "validation"]]


## 8. DV01 계산

DV01은 금리가 1bp 상승할 때 발생하는 가격감소액을 양수로 표시합니다.


In [ ]:
def calculate_dv01(bond, curve):
    base_price = bond_price(
        bond["face_value"], bond["coupon_rate"], bond["maturity"],
        bond["frequency"], curve, bond["credit_spread_bp"]
    )

    shifted_curve = curve.copy()
    shifted_curve["yield"] = shifted_curve["yield"] + 0.0001

    shifted_price = bond_price(
        bond["face_value"], bond["coupon_rate"], bond["maturity"],
        bond["frequency"], shifted_curve, bond["credit_spread_bp"]
    )
    return base_price - shifted_price

bonds["DV01"] = bonds.apply(lambda row: calculate_dv01(row, curve), axis=1)
bonds[["bond_id", "maturity", "reference_price", "DV01"]]


## 9. Effective Duration 계산

금리를 위아래로 1bp 이동시켜 채권가격을 재평가하는 방식으로 Effective Duration을 계산합니다.


In [ ]:
def effective_duration(bond, curve, shock_bp=1):
    shock = shock_bp / 10000
    base_price = bond_price(
        bond["face_value"], bond["coupon_rate"], bond["maturity"],
        bond["frequency"], curve, bond["credit_spread_bp"]
    )
    curve_up = curve.copy(); curve_up["yield"] = curve_up["yield"] + shock
    curve_down = curve.copy(); curve_down["yield"] = curve_down["yield"] - shock
    price_up = bond_price(bond["face_value"], bond["coupon_rate"], bond["maturity"], bond["frequency"], curve_up, bond["credit_spread_bp"])
    price_down = bond_price(bond["face_value"], bond["coupon_rate"], bond["maturity"], bond["frequency"], curve_down, bond["credit_spread_bp"])
    return (price_down - price_up) / (2 * base_price * shock)

bonds["effective_duration"] = bonds.apply(lambda row: effective_duration(row, curve), axis=1)
bonds[["bond_id", "maturity", "DV01", "effective_duration"]]


## 10. 금리 Stress Test

Yield Curve 전체가 +100bp, -100bp 평행 이동한다고 가정하여 Full Revaluation을 수행합니다.


In [ ]:
def stress_price(bond, curve, shock_bp):
    stressed_curve = curve.copy()
    stressed_curve["yield"] = stressed_curve["yield"] + shock_bp / 10000
    return bond_price(
        bond["face_value"], bond["coupon_rate"], bond["maturity"],
        bond["frequency"], stressed_curve, bond["credit_spread_bp"]
    )

stress_rows = []
for _, bond in bonds.iterrows():
    base = bond["reference_price"]
    up100 = stress_price(bond, curve, 100)
    down100 = stress_price(bond, curve, -100)
    stress_rows.append({
        "bond_id": bond["bond_id"],
        "base_price": base,
        "plus_100bp_price": up100,
        "plus_100bp_return_pct": (up100 / base - 1) * 100,
        "minus_100bp_price": down100,
        "minus_100bp_return_pct": (down100 / base - 1) * 100
    })

stress_df = pd.DataFrame(stress_rows)
stress_df


## 11. Stress Test 시각화


In [ ]:
plot_df = stress_df.set_index("bond_id")[["plus_100bp_return_pct", "minus_100bp_return_pct"]]
ax = plot_df.plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("Price Change (%)")
ax.set_title("Interest Rate Stress Test")
ax.axhline(0, linewidth=1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. 결과 해석

다음 질문에 직접 답해 README의 결과 부분을 작성합니다.

1. 세 채권 중 DV01이 가장 큰 채권은 무엇인가?
2. 만기가 길어질수록 Effective Duration은 어떤 방향으로 변했는가?
3. +100bp 충격에서 가장 큰 손실을 보인 채권은 무엇인가?
4. BOND_B와 BOND_C가 REVIEW로 분류된 이유는 무엇인가?
5. REVIEW가 곧 평가엔진의 오류를 의미하지 않는 이유는 무엇인가?
6. 실제 프로젝트로 발전시키려면 어떤 시장데이터와 모델을 추가해야 하는가?

이 여섯 질문을 본인이 설명할 수 있어야 면접에서 프로젝트를 방어할 수 있습니다.


## 13. 프로젝트 한계

- 현재 Yield Curve는 예시 데이터
- 단순 선형보간 사용
- Credit Spread를 만기 전 구간에 동일하게 적용
- Accrued Interest, Clean Price, Day Count Convention 미반영
- 유동성 프리미엄 미반영
- 가상 Engine Price 사용
- 실제 평가사 내부모형 및 검증기준과 동일하지 않음

따라서 이 프로젝트는 실제 평가가격 제공이 목적이 아니라, 채권 평가와 가격 검증의 핵심 구조를 이해하고 재현하기 위한 학부 수준의 프로젝트입니다.
